# MultipleMoviesStim video (NPZ)

Loads `MultipleMoviesStim_1_tree.npz` (frames written by `scripts/process_avi_to_numpy.py`) and plays it in the notebook with a Matplotlib JS HTML animation, same pattern as `particle_orbit.ipynb` (`FuncAnimation` → `to_jshtml`).

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

In [ ]:
NPZ_PATH = Path("/Users/ruimeng/data/MultipleMoviesStim_1_tree.npz")

# Full array is large; subsample for faster load / smaller HTML embed.
FRAME_STRIDE = 1
MAX_FRAMES: int | None = None  # e.g. 300 to cap animation length

with np.load(NPZ_PATH, allow_pickle=True) as z:
    frames = np.asarray(z["frames"])
    fps_arr = np.asarray(z["fps"])
    shape_arr = np.asarray(z["shape"]) if "shape" in z.files else None

if FRAME_STRIDE > 1 or MAX_FRAMES is not None:
    idx = np.arange(0, frames.shape[0], FRAME_STRIDE)
    if MAX_FRAMES is not None:
        idx = idx[:MAX_FRAMES]
    frames = np.asarray(frames[idx])

fps = float(fps_arr.reshape(-1)[0]) if fps_arr.size else 0.0
print("frames", frames.shape, frames.dtype)
print("fps", fps)
if shape_arr is not None:
    print("stored shape metadata", shape_arr)

In [ ]:
def _frame_for_imshow(f: np.ndarray) -> np.ndarray:
    if f.ndim == 2:
        return f
    if f.ndim == 3 and f.shape[-1] == 1:
        return f[..., 0]
    return f


def animate_video_frames(frames: np.ndarray, interval: float = 80.0):
    """HTML-style animation like `animate_particle_timeseries` in particle_orbit.ipynb."""
    T = int(frames.shape[0])
    f0 = _frame_for_imshow(np.asarray(frames[0]))
    w = max(int(f0.shape[1]), 1)
    fig_h = max(3.0, f0.shape[0] / w * 10)
    fig, ax = plt.subplots(figsize=(10, fig_h))
    im = ax.imshow(f0, aspect="equal", interpolation="nearest")
    ax.axis("off")

    def update(frame: int):
        im.set_data(_frame_for_imshow(np.asarray(frames[frame])))
        ax.set_title(f"t = {frame}")
        return (im,)

    return animation.FuncAnimation(fig, update, frames=T, interval=interval, blit=False)


interval_ms = (1000.0 / fps) if fps > 1e-6 else 80.0
anim = animate_video_frames(frames, interval=interval_ms)
plt.close(anim._fig)
HTML(anim.to_jshtml())